# Stillwater 4-Tetromino Project Sibling

This notebook creates a compact SewerTris sibling from `example_02_Stillwater_sewertris_project.ipynb`. The base project already defines the Stillwater domain, modeling parameters, SWMM options, and scenario settings. Here we only declare the change: use the four-tetromino set `I_O_T_S_only`.

`clone_sibling(...)` copies only the reusable artifacts from the parent project. Because the tetromino set changes the urban layout, the sibling reuses the domain artifacts and reruns the dependent physical and SWMM workflow using the parameters recorded in the parent project manifest.


## Setup

Load the refactored package and the completed Example 02 project. If you are editing package code while working in this notebook, rerunning this cell reloads the local modules.


In [ ]:
from pathlib import Path
import importlib
import sys

import geopandas as gpd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "sewertris").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

EXAMPLES_DIR = PROJECT_ROOT / "examples"
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import sewertris

for _module_name in [
    "sewertris._deps",
    "sewertris.domain",
    "sewertris.layout",
    "sewertris.roads",
    "sewertris.topography",
    "sewertris.sewer_network",
    "sewertris.hydrology",
    "sewertris.design",
    "sewertris.swmm",
    "sewertris.plots",
    "sewertris.project",
    "sewertris",
]:
    if _module_name in sys.modules:
        importlib.reload(sys.modules[_module_name])

import sewertris as sp

base_project_path = EXAMPLES_DIR / "output_example_2_project"

base_project = sp.SewerTrisProject.load(base_project_path)
print("Base project:", base_project.project_file)


## Clone Sibling

Create the sibling project and record its lineage. The only model change is the tetromino set. The clone keeps the parent domain files and marks the workflow to rerun from the tetromino/layout stage onward.


In [ ]:
project = base_project.clone_sibling(
    EXAMPLES_DIR / "output_example_3_project",
    name="Stillwater 4-Tetromino SewerTris Project",
    changes={"tetromino_set": "I_O_T_S_only"},
)

print("Sibling project:", project.project_file)
print("Rerun from:", project.metadata["lineage"]["rerun_from"])
print("Copied artifacts:", project.metadata["lineage"]["copied_artifacts"])


## Run Sibling

Rerun the dependent workflow using parameters from the parent project. This call regenerates the layout, roads, land use, DEM, sewer network, flow predesign, pipe design, base SWMM file, and named SWMM scenario.


In [ ]:
results = project.rerun_from_parent_parameters(
    base_project,
    tetromino_set="I_O_T_S_only",
    scenario_name="bwf_gwi_rdii",
    run_flow_components=True,
)

scenario = results["scenario"]
df = results["flow_components"]

print("Sibling metadata:", project.project_file)
print("Scenario input:", scenario.swmm_inp_path)
print("Flow components:", scenario.flows_path)


## Review Outputs

Use the standard plots to inspect the sibling. These are optional diagnostics; the model artifacts and lineage are already saved in `output_example_3_project`.


In [ ]:
sp.plot_filled_board_shapefile(project.layout_blocks_path)

sp.plot_roads(
    road_lines=results["road_lines"],
    road_buffer=results["road_buffer"],
    crs=results["crs"],
    title="Stillwater 4-tetromino road network",
)

roads_gdf = gpd.read_file(project.road_polygons_path)
blocks_gdf = gpd.read_file(project.blocks_path)
if roads_gdf.crs != blocks_gdf.crs:
    roads_gdf = roads_gdf.to_crs(blocks_gdf.crs)
sp.plot_blocks_landuse(
    blocks_gdf=blocks_gdf,
    roads_gdf=roads_gdf,
    landuse_col="land_use",
    title="Stillwater 4-tetromino land use",
)

sp.plot_dem_tif(project.dem_path, title="Stillwater 4-tetromino modified DEM", hillshade=False)

sp.plot_sewer_network_all(
    manholes=project.state["manholes"],
    main_pipes=project.state["main_path"],
    secondary_pipes=project.state["secondary_pipes"],
    tertiary_pipes=project.state["tertiary_pipes"],
    unresolved=project.state["tertiary_unconnected"],
    road_buffer=project.state["road_buffer"],
    title="Stillwater 4-tetromino sewer network",
)

sp.plot_final_design_color_by_diameter(
    pipes_path=project.pipes_path,
    manholes_path=project.manholes_path,
    blocks_path=project.blocks_path,
    diameter_field="diameter_mm",
    manhole_color_field="elevation",
    linewidth=1.8,
)


## Flow Decomposition

Plot and save the sibling flow components using the parent simulation dates recorded in the SWMM options.


In [ ]:
options = base_project.step_parameters("10_dynamic_flow_input_definition_base_model").get("options_dict", {})
start = f"{options.get('START_DATE', '01/01/1990')} {options.get('START_TIME', '00:00:00')}"
end = f"{options.get('END_DATE', '01/10/1990')} {options.get('END_TIME', '00:00:00')}"

sp.plot_flow_components_v2(df, start=start, end=end)
project.save()
print("Project metadata:", project.project_file)
print("Parent project:", project.metadata["lineage"]["parent_project_file"])


## Compare Base vs Sibling

Create side-by-side comparison plots between the original Stillwater project and the 4-tetromino sibling. The comparison reads the completed project artifacts directly, so no model inputs need to be redefined here.


In [ ]:
comparison_figures = sp.plot_two_models(
    "all",
    base_project,
    project,
    labels=("Original Stillwater", "4-tetromino sibling"),
    scenario_name="bwf_gwi_rdii",
    start=start,
    end=end,
    diameter_field="diameter_mm",
    manhole_color_field="elevation",
)
